In [5]:
import requests
import pathlib
import xml.etree.ElementTree as ET
import json
import time
import string
import sys
import pandas as pd
from deutsche_bahn_api import ApiAuthentication
from pathlib import Path
import os
import numpy as np
import re
from datetime import datetime, timedelta
from math import radians, cos, sin, asin, sqrt


BASE_DB_API = "https://apis.deutschebahn.com/db-api-marketplace/apis/"
STOP_PLACES_URL = "https://apis.deutschebahn.com/db-api-marketplace/apis/ris-stations/v1/stop-places"
TIMETABLES_V1_URL = BASE_DB_API + "timetables/v1"
RIS_STATION_URL = BASE_DB_API + "ris-station/v1/stations/"
ROOT_DIR = Path(os.getcwd())
DATA_DIR = ROOT_DIR / "data"
RAW_DATA_DIR = ROOT_DIR / "Raw_Data"

DB_CLIENT_ID = "a9f83c55d26c3ee7f48f4ce887ec2a57"
DB_API_KEY = "422cac21a0a83876c75efb8806589ea0"
PKP_API_KEY = "KMjLifnR-a6RGlVgs36-OGG82nZ2gZRuiySF_y-tWSbUMX4LNS3JfBk1hli1B59YIXfdIrIy3ZvwUpkrMueAeA"

header = {
    "DB-Client-Id": DB_CLIENT_ID,
    "DB-Api-Key": DB_API_KEY,
    "accept": "application/xml"
}

header2 = {
    "DB-Client-Id": DB_CLIENT_ID,
    "DB-Api-Key": DB_API_KEY,
    "accept": "application/json"
}

def get_all_stations(country_code="DE"):
    params = {"state": country_code}
    
    response = requests.get(RIS_STATION_URL, headers=header, params=params)
    if response.status_code == 200:
        return response.json().get('stations', [])
    else:   
        print(f"Error: {response.status_code}")
        return []

def haversine(lat1, lon1, lat2, lon2):
    """ Calculate distance in km between two points """
    R = 6371 # Earth radius
    dLat = radians(lat2 - lat1)
    dLon = radians(lon2 - lon1)
    a = sin(dLat/2)**2 + cos(radians(lat1)) * cos(radians(lat2)) * sin(dLon/2)**2
    return 2 * R * asin(sqrt(a))

# api credentials validation
api_authentication = ApiAuthentication( DB_CLIENT_ID, DB_API_KEY)
success:bool = api_authentication.test_credentials()
success

/Users/rafael/projects/DOPP_groupB_2025W/.venv/lib/python3.13/site-packages/mpu/string.py:16: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


True

#  Railway stations selection & filtering

First, match train station with selected cities from simplemaps ( Processed_Data/cities_500K). Later for matched cities retrieve required timetables.

Matching should be done geospatially

In [6]:
PROCESSED_DATA_DIR = ROOT_DIR / "Processed_Data"

cities_data = pd.read_csv(PROCESSED_DATA_DIR / "cities_500K.csv")
cities_data.rename(columns={"name_city":"name", "lat_city":"latitude", "lng_city":"longitude","country":"iso_code"}, inplace=True)
cities_data.head()

,name,latitude,longitude,iso_code,population,is_capital
0,Vienna,48.2083,16.3725,AT,1973403.0,True
1,Brussels,50.8467,4.3525,BE,1235192.0,True
2,Antwerp,51.2178,4.4003,BE,536079.0,False
3,Sofia,42.7000,23.3300,BG,1383435.0,True
4,Prague,50.0875,14.4214,CZ,1357326.0,True


In [11]:
RIS_STATION_URL = "https://apis.deutschebahn.com/db-api-marketplace/apis/ris-stations/v1/stations"
header_ris = {
    "DB-Client-Id": DB_CLIENT_ID,
    "DB-Api-Key": DB_API_KEY,
    "accept": "application/vnd.de.db.ris+json"
}

res = requests.get(url=RIS_STATION_URL, headers=header_ris, params={"countryCode":"AT"})
res.text

'{"offset":0,"limit":100,"total":5703,"stations":[{"stationID":"1","names":{"DE":{"name":"Aachen Hbf"}},"metropolis":{},"address":{"street":"Bahnhofstr.","houseNumber":"2a","postalCode":"52064","city":"Aachen","state":"Nordrhein-Westfalen","country":"DE"},"stationCategory":"CATEGORY_2","availableTransports":[],"availableLocalServices":[],"transportAssociations":[],"owner":{"name":"DB InfraGO Personenbahnhöfe","organisationalUnit":{"id":4,"name":"RB West","nameShort":"RB West"}},"countryCode":"DE","state":"NW","timeZone":"Europe/Berlin","position":{"longitude":6.091499,"latitude":50.7678},"validFrom":"2018-12-31T23:00:00Z","mobilityServiceStaffOnSite":true},{"stationID":"1000","names":{"DE":{"name":"Burkhardswalde-Maxen"}},"metropolis":{},"address":{"street":"Gesundbrunnen","houseNumber":"60c","postalCode":"01809","city":"Müglitztal-Burkhardswalde","state":"Sachsen","country":"DE"},"stationCategory":"CATEGORY_7","availableTransports":[],"availableLocalServices":[],"transportAssociations

Testing stop-places accesspoint

In [12]:
response = requests.get(url=f"{STOP_PLACES_URL}/by-name/Warsaw?sortBy=RELEVANCE&onlyActive=true&withSynonyms=true&limit=3", headers=header_ris)
response.text

'{"stopPlaces":[{"evaNumber":"5100065","groupMembers":[],"names":{"DE":{"nameLong":"Warszawa Centralna","synonyms":[]}},"replacementTransportsAvailable":false,"availableTransports":["INTERCITY_TRAIN","INTER_REGIONAL_TRAIN"],"position":{"longitude":21.003234,"latitude":52.22886}},{"evaNumber":"5100067","groupMembers":[],"names":{"DE":{"nameLong":"Warszawa Zachodnia","synonyms":[]}},"replacementTransportsAvailable":false,"availableTransports":["INTERCITY_TRAIN","INTER_REGIONAL_TRAIN"],"position":{"longitude":20.965247,"latitude":52.219972}},{"evaNumber":"5100066","groupMembers":[],"names":{"DE":{"nameLong":"Warszawa Wschodnia","synonyms":[]}},"replacementTransportsAvailable":false,"availableTransports":["INTERCITY_TRAIN","INTER_REGIONAL_TRAIN"],"position":{"longitude":21.052335,"latitude":52.251548}}]}'

Using RIS:stations retrieve all cities main stations

In [13]:
def get_stations_api_stop_places(cities : pd.DataFrame, 
                      base_url: str = STOP_PLACES_URL,  
                      limit: int = 3) -> pd.DataFrame:
    """
    Retrieves station information from the Deutsche Bahn API based on the station name.
    
    Keyword arguments:
    cities -- DataFrame containing city information
    base_url -- Base URL for the API
    limit -- Maximum number of stations to retrieve per city
    Return: DataFrame of stations for given cities
    """
    retrieved_stations = pd.DataFrame(columns=['city_name', 'eva_id', 'station_name', 'latitude', 'longitude'])
    for _, row in cities.iterrows():
        city_name = row['name']
        response = None
        params = {
            "sortBy": "RELEVANCE",
            "onlyActive": "true",
            "withSynonyms": "true",
            "latitude": row['latitude'],
            "longitude": row['longitude'],
            "limit": limit
        }
        success = False
        retries = 0

        while not success and retries < 3:
            response = requests.get(url=f"{base_url}/by-name/{city_name}", params=params, headers=header_ris)

            if response.status_code == 429:
                print(f"Rate limit reached. Sleeping for 1s...")
                time.sleep(1)
                retries += 1
                continue

            if response.status_code == 200:
                stations = response.json().get('stopPlaces', [])
                print(f"City: {city_name}, Stations Found: {len(stations)}")
                for station in stations:
                    latitude = float(station.get('position').get('latitude'))
                    longitude = float(station.get('position').get('longitude'))

                    if haversine(row['latitude'], row['longitude'], latitude, longitude) > 10:
                        print(f"Skipping station {station.get('names').get('DE').get('nameLong')} due to distance.")
                        continue

                    retrieved_stations = pd.concat([retrieved_stations, pd.DataFrame({
                        'city_name': [city_name],
                        'eva_id': [str(station.get('evaNumber'))],
                        'station_name': [str(station.get('names').get('DE').get('nameLong'))],
                        'latitude': [float(station.get('position').get('latitude'))],
                        'longitude': [float(station.get('position').get('longitude'))]
                    })])
                success = True
            else:
                print(f"Error retrieving stations for city {city_name}: {response.status_code}")
                break

            time.sleep(0.09)  # To respect API rate limits

    return retrieved_stations

In [14]:
test_cities = cities_data.iloc[np.random.choice(cities_data.shape[0], 5, replace=False)]
test_cities

,name,latitude,longitude,iso_code,population,is_capital
59,Bratislava,48.1439,17.1097,SK,475503.0,True
49,Warsaw,52.2300,21.0111,PL,1860281.0,True
17,Hannover,52.3667,9.7167,DE,545045.0,False
44,Riga,56.9489,24.1064,LV,660187.0,True
42,Vilnius,54.6872,25.2800,LT,581475.0,True


In [15]:
output  = get_stations_api_stop_places(test_cities, limit=3)

City: Bratislava, Stations Found: 3


/var/folders/pj/dyqj7jwx6r76zl2c1w7dvc7h0000gn/T/ipykernel_9906/3370103374.py:48: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  retrieved_stations = pd.concat([retrieved_stations, pd.DataFrame({


City: Warsaw, Stations Found: 3
City: Hannover, Stations Found: 3
City: Riga, Stations Found: 3
Skipping station Küssnacht am Rigi due to distance.
Skipping station Bad Godesberg Rigal'sche Wiese, Bonn due to distance.
Skipping station Rigaer Straße, Rostock due to distance.
City: Vilnius, Stations Found: 1
Skipping station Vilniuser Straße, Erfurt due to distance.


For cities above 500K we select at least 3 station if possible as timetables for those stations might contain most crutial connections

In [16]:
cities_stations = get_stations_api_stop_places(cities_data, limit=1)

City: Vienna, Stations Found: 1


/var/folders/pj/dyqj7jwx6r76zl2c1w7dvc7h0000gn/T/ipykernel_9906/3370103374.py:48: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  retrieved_stations = pd.concat([retrieved_stations, pd.DataFrame({


City: Brussels, Stations Found: 1
City: Antwerp, Stations Found: 1
City: Sofia, Stations Found: 1
Skipping station Sofie-Hammer-Straße, Osnabrück due to distance.
City: Prague, Stations Found: 1
City: Berlin, Stations Found: 1
City: Stuttgart, Stations Found: 1
City: Munich, Stations Found: 1
City: Hamburg, Stations Found: 1
City: Cologne, Stations Found: 1
City: Frankfurt, Stations Found: 1
City: Düsseldorf, Stations Found: 1
City: Leipzig, Stations Found: 1
City: Dortmund, Stations Found: 1
City: Essen, Stations Found: 1
City: Bremen, Stations Found: 1
City: Dresden, Stations Found: 1
City: Hannover, Stations Found: 1
City: Nuremberg, Stations Found: 1
City: Duisburg, Stations Found: 1
City: Copenhagen, Stations Found: 1
City: Tallinn, Stations Found: 1
Skipping station Tallinner Straße, Schwerin (Meckl) due to distance.
City: Madrid, Stations Found: 1
Skipping station Madrider Ring, Würzburg due to distance.
City: Barcelona, Stations Found: 1
City: Valencia, Stations Found: 1
Skippi

In [17]:
cities_stations

,city_name,eva_id,station_name,latitude,longitude
0,Vienna,8103000,Wien Hbf,48.185101,16.377113
0,Brussels,8800004,Bruxelles Midi,50.835376,4.335694
0,Antwerp,8800007,Antwerpen Centraal,51.215811,4.421168
0,Prague,5400014,Praha hl.n.,50.083062,14.436039
0,Berlin,8011160,Berlin Hbf,52.525592,13.369545
0,Stuttgart,8000096,Stuttgart Hbf,48.784780,9.182757
0,Munich,8000261,München Hbf,48.140232,11.558335
0,Hamburg,8002549,Hamburg Hbf,53.552736,10.006909
0,Cologne,8003368,Köln Messe/Deutz,50.940874,6.975001
0,Frankfurt,8000105,Frankfurt(Main)Hbf,50.106682,8.662828


In [18]:
cities_stations[cities_stations['city_name'] == "Prague" ]

,city_name,eva_id,station_name,latitude,longitude
0,Prague,5400014,Praha hl.n.,50.083062,14.436039


Found Train Stations for given cities

first cities which had no stations at DB API

In [19]:
cities_no_station = pd.merge(cities_data, cities_stations, how='outer',left_on=['name'], right_on=['city_name'], indicator=True).query('_merge == "left_only"')
cities_no_station.reset_index(inplace=True)
cities_no_station

,index,name,latitude_x,longitude_x,iso_code,population,is_capital,city_name,eva_id,station_name,latitude_y,longitude_y,_merge
0,2,Athens,37.9842,23.7281,GR,643452.0,True,NaN,NaN,NaN,NaN,NaN,left_only
1,9,Bucharest,44.4325,26.1039,RO,1716961.0,True,NaN,NaN,NaN,NaN,NaN,left_only
2,20,Gothenburg,57.7075,11.9675,SE,607882.0,False,NaN,NaN,NaN,NaN,NaN,left_only
3,23,Helsinki,60.1708,24.9375,FI,664921.0,True,NaN,NaN,NaN,NaN,NaN,left_only
4,26,Lisbon,38.7253,-9.1500,PT,548703.0,True,NaN,NaN,NaN,NaN,NaN,left_only
5,30,Madrid,40.4169,-3.7033,ES,3266126.0,True,NaN,NaN,NaN,NaN,NaN,left_only
6,34,Málaga,36.7194,-4.4200,ES,586384.0,False,NaN,NaN,NaN,NaN,NaN,left_only
7,35,Naples,40.8333,14.2500,IT,913462.0,False,NaN,NaN,NaN,NaN,NaN,left_only
8,40,Riga,56.9489,24.1064,LV,660187.0,True,NaN,NaN,NaN,NaN,NaN,left_only
9,43,Sevilla,37.3900,-5.9900,ES,684025.0,False,NaN,NaN,NaN,NaN,NaN,left_only


In [20]:
cities_with_stations = pd.merge(cities_data, cities_stations, how='inner',left_on=['name'], right_on=['city_name'])
cities_with_stations

,name,latitude_x,longitude_x,iso_code,population,is_capital,city_name,eva_id,station_name,latitude_y,longitude_y
0,Vienna,48.2083,16.3725,AT,1973403.0,True,Vienna,8103000,Wien Hbf,48.185101,16.377113
1,Brussels,50.8467,4.3525,BE,1235192.0,True,Brussels,8800004,Bruxelles Midi,50.835376,4.335694
2,Antwerp,51.2178,4.4003,BE,536079.0,False,Antwerp,8800007,Antwerpen Centraal,51.215811,4.421168
3,Prague,50.0875,14.4214,CZ,1357326.0,True,Prague,5400014,Praha hl.n.,50.083062,14.436039
4,Berlin,52.5200,13.4050,DE,3755251.0,True,Berlin,8011160,Berlin Hbf,52.525592,13.369545
5,Stuttgart,48.7775,9.1800,DE,632865.0,False,Stuttgart,8000096,Stuttgart Hbf,48.784780,9.182757
6,Munich,48.1375,11.5750,DE,1512491.0,False,Munich,8000261,München Hbf,48.140232,11.558335
7,Hamburg,53.5500,10.0000,DE,1892122.0,False,Hamburg,8002549,Hamburg Hbf,53.552736,10.006909
8,Cologne,50.9364,6.9528,DE,1084831.0,False,Cologne,8003368,Köln Messe/Deutz,50.940874,6.975001
9,Frankfurt,50.1106,8.6822,DE,773068.0,False,Frankfurt,8000105,Frankfurt(Main)Hbf,50.106682,8.662828


# Rertieve Timetables for existing city stations

currently timetables for DB offers limited data. Other timetables are required to be used

In [122]:
header_timetables = {
    "DB-Client-Id": DB_CLIENT_ID,
    "DB-Api-Key": DB_API_KEY,
    "accept": "application/xml"
}


def fetch_hourly_plan(eva_no, date_str, hour_str):
    """
    Fetches the planned timetable for a specific station, date (YYMMDD), and hour (HH).
    """
    url = f"{TIMETABLES_V1_URL}/plan/{eva_no}/{date_str}/{hour_str}"
    success = False
    rertries = 0
    plan_hour = ""

    
    while not success and rertries < 3:
        response = requests.get(url, headers=header_timetables)
        status_code  = response.status_code

        if status_code == 200:
            success = True
            retries = 0
            plan_hour =  response.text
        elif status_code == 404 or status_code == 401:
            print("Access Error: Check API")
            break 
        elif status_code == 429:
            time.sleep(1)
        retries += 1
            
    return plan_hour

def parse_hour_plan(hour_xml:str):
    pass

# test run 
fetch_hourly_plan("8103000","251229","12")


'<?xml version=\'1.0\' encoding=\'UTF-8\'?><timetable station=\'Wien Hbf\'><s id="-641886370829105986-2512291130-3"><tl f="F" t="p" o="81" c="EC" n="204"/><ar pt="2512291158" pp="12A-B" fb="EC 204" ppth="Wien Westbahnhof|Wien Meidling"/><dp pt="2512291210" pp="12A-B" fb="EC 204" pde="Krakow Glowny" ppth="Breclav|Hodonin|Stare Mesto u Uherského Hradiste|Otrokovice|Prerov|Hranice na Morave|Ostrava-Svinov|Ostrava hl.n.|Bohumin"/></s><s id="-721807130937592434-2512291213-1"><tl f="F" t="p" o="81" c="ICE" n="90"/><dp pt="2512291213" pp="8A-B" fb="ICE 90" ppth="Wien Meidling|St.Pölten Hbf|Linz Hbf|Passau Hbf|Plattling|Regensburg Hbf|Nürnberg Hbf|Coburg|Erfurt Hbf|Leipzig Hbf|Lutherstadt Wittenberg Hbf|Berlin Südkreuz|Berlin Hbf|Berlin Gesundbrunnen"/></s><s id="-6130228582536441801-2512290618-10"><tl f="F" t="p" o="51" c="EC" n="203"/><ar pt="2512291149" pp="6A-B" fb="EC 203" pde="Krakow Glowny" ppth="Bohumin|Ostrava hl.n.|Ostrava-Svinov|Hranice na Morave|Prerov|Otrokovice|Stare Mesto u Uher

# Using v6.db.transport.rest

In [25]:
from typing import Optional, List, Dict, Any

BASE_URL = "https://v6.db.transport.rest"
RATE_LIMIT_SLEEP = 1.0  # Seconds to sleep between requests to respect 100 req/min
MAX_RETRIES = 3

def fetch_route_raw(origin_id: str, dest_id: str, departure_dt: datetime) -> Optional[Dict]:
    """
    Helper 1: Performs the actual API request for a specific date/time.
    Handles rate limiting and retries.
    """
    params = {
        "from": origin_id,
        "to": dest_id,
        "departure": departure_dt.isoformat(),
        "results": 1,           # We just need the fastest connection for this slot
        "national": "true",     # Prefer long-distance
        "nationalExpress": "true",
        # "profile": "dbweb"      # Web profile often has better international data
    }

    attempts = 0
    while attempts < MAX_RETRIES:
        try:
            response = requests.get(f"{BASE_URL}/journeys", params=params, timeout=10)
            
            if response.status_code == 200:
                time.sleep(RATE_LIMIT_SLEEP) # Polite wait
                return response.json()
            
            elif response.status_code == 429:
                wait = 2 * (attempts + 1)
                print(f"    ⚠️ Rate limit (429). Sleeping {wait}s...")
                time.sleep(wait)
                attempts += 1
            else:
                print(f"    ❌ Error {response.status_code}")
                return None
                
        except Exception as e:
            print(f"    ❌ Request failed: {e}")
            attempts += 1
            time.sleep(1)
            
    return None

route_warsaw_berlin  = fetch_route_raw("5100065", "8011160", datetime.now())

In [ ]:
def extract_journey_metrics(journey_data: Dict) -> Optional[Dict]:
    """
    Helper 2: Parses raw JSON to extract duration, transfers, and train names.
    """
    journeys = journey_data.get('journeys', [])
    if not journeys:
        return None

    # We asked for 1 result, so take the first
    best_journey = journeys[0]
    legs = best_journey.get('legs', [])
    
    if not legs:
        return None

    # Calculate timestamps and duration
    dep_str = legs[0]['departure']
    arr_str = legs[-1]['arrival']
    t_dep = datetime.fromisoformat(dep_str)
    t_arr = datetime.fromisoformat(arr_str)
    duration_minutes = (t_arr - t_dep).total_seconds() / 60

    # Extract train names for context (e.g., "ICE 100")
    trains = [
        leg.get('line', {}).get('name', '') 
        for leg in legs 
        if leg.get('mode') == 'train'
    ]

    return {
        "departure_time": dep_str,
        "arrival_time": arr_str,
        "duration_minutes": int(duration_minutes),
        "transfers": len(legs) - 1,
        "trains": ", ".join(filter(None, trains)),
        "origin_name": legs[0]['origin']['name'],
        "destination_name": legs[-1]['destination']['name']
    }

extract_journey_metrics(route_warsaw_berlin)



{'departure_time': '2025-12-30T22:16:00+01:00',
 'arrival_time': '2025-12-31T06:14:00+01:00',
 'duration_minutes': 478,
 'transfers': 0,
 'trains': '',
 'origin_name': 'Warszawa Centralna',
 'destination_name': 'Berlin Hbf'}

In [28]:
def get_best_weekly_connection(origin_id: str, dest_id: str, start_date: datetime) -> Optional[Dict]:
    """
    Helper 3: Loops through 7 days to find the minimum duration for a pair.
    """
    best_metrics = None
    min_duration = float('inf')

    # Check the same time for the next 7 days (e.g., every morning at 08:00)
    # This accounts for weekends vs weekdays schedules
    for day_offset in range(7):
        current_date = start_date + timedelta(days=day_offset)
        print(f"  Checking {current_date.strftime('%Y-%m-%d')}...", end="\r")
        
        raw_data = fetch_route_raw(origin_id, dest_id, current_date)
        
        if raw_data:
            metrics = extract_journey_metrics(raw_data)
            if metrics:
                # Update if this day offers a faster trip
                if metrics['duration_minutes'] < min_duration:
                    min_duration = metrics['duration_minutes']
                    best_metrics = metrics
                    # Add IDs back to the dict for the final dataframe
                    best_metrics['origin_id'] = origin_id
                    best_metrics['destination_id'] = dest_id
                    best_metrics['best_day'] = current_date.strftime('%A') # e.g. "Monday"

    return best_metrics

get_best_weekly_connection("5100065", "8011160", datetime.now() - timedelta(days=7))

{'departure_time': '2025-12-23T22:16:00+01:00',
 'arrival_time': '2025-12-24T06:14:00+01:00',
 'duration_minutes': 478,
 'transfers': 0,
 'trains': '',
 'origin_name': 'Warszawa Centralna',
 'destination_name': 'Berlin Hbf',
 'origin_id': '5100065',
 'destination_id': '8011160',
 'best_day': 'Tuesday'}

In [25]:
from itertools import permutations


def get_next_representative_weekday():
    """
    Finds the next Tuesday or Wednesday. 
    Mid-week days have the most consistent 'standard' schedules.
    """
    now = datetime.now()
    # 0=Mon, 1=Tue, 2=Wed...
    days_ahead = (1 - now.weekday() + 7) % 7 # Target Tuesday
    if days_ahead == 0: days_ahead = 7 # If today is Tuesday, get next week
    
    target_date = now + timedelta(days=days_ahead)
    # Set to 08:00 AM - Peak morning traffic usually has the best connections
    return target_date.replace(hour=8, minute=0, second=0, microsecond=0)

def fetch_fastest_connection(origin_id: str, dest_id: str) -> Optional[Dict]:
    """
    Makes a SINGLE smart request to find the fastest connection.
    Retrieves 5 options and picks the minimum duration.
    """
    target_date = get_next_representative_weekday()
    
    params = {
        "from": origin_id,
        "to": dest_id,
        "departure": target_date.isoformat(),
        "results": 5,           # Get 5 options in ONE request
        "national": "true",     # Prefer High Speed
        "nationalExpress": "true",
        "transfers": 4          # Allow complex routes if they are faster
    }

    # Retry logic for stability
    for attempt in range(3):
        try:
            response = requests.get(f"{BASE_URL}/journeys", params=params, timeout=10)
            
            if response.status_code == 200:
                data = response.json()
                journeys = data.get('journeys', [])
                
                if not journeys:
                    return None

                # Optimization: Process all 5 results locally to find the minimum
                min_duration = float('inf')
                best_journey = None

                for journey in journeys:
                    if not journey.get('legs'): continue
                    
                    dep = datetime.fromisoformat(journey['legs'][0]['departure'])
                    arr = datetime.fromisoformat(journey['legs'][-1]['arrival'])
                    duration = (arr - dep).total_seconds() / 60
                    
                    if duration < min_duration:
                        min_duration = duration
                        best_journey = journey
                        best_journey['calculated_duration'] = int(duration)

                # Extract details from the winner
                legs = best_journey['legs']
                train_names = [l.get('line', {}).get('name', '') for l in legs if l.get('mode') == 'train']

                time.sleep(RATE_LIMIT_SLEEP) # Respect limits
                return {
                    "origin_id": origin_id,
                    "destination_id": dest_id,
                    "origin_name": legs[0]['origin']['name'],
                    "destination_name": legs[-1]['destination']['name'],
                    "min_duration_minutes": best_journey['calculated_duration'],
                    "transfers": len(legs) - 1,
                    "trains": ", ".join(filter(None, train_names)),
                    "check_date": target_date.strftime("%Y-%m-%d")
                }

            elif response.status_code == 429:
                time.sleep(3 * (attempt + 1)) # Backoff
            elif response.status_code >= 500:
                time.sleep(2)
            else:
                return None
                
        except Exception as e:
            time.sleep(1)

    return None

def get_efficient_network_data(eva_ids: List[str]) -> pd.DataFrame:
    """
    Main Runner.
    Reduced Complexity: O(N * (N-1)) requests instead of O(N * (N-1) * 7).
    """
    # Use permutations because A->B might be slightly different than B->A 
    # (e.g. connections matching up)
    pairs = list(permutations(eva_ids, 2))
    
    print(f"🚀 Optimized Search: {len(eva_ids)} stations -> {len(pairs)} routes.")
    print(f"📅 Using representative weekday: {get_next_representative_weekday().strftime('%A, %Y-%m-%d')}")
    
    results = []
    
    for i, (origin, dest) in enumerate(pairs):
        print(f"[{i+1}/{len(pairs)}] {origin} -> {dest}...", end=" ", flush=True)
        
        data = fetch_fastest_connection(origin, dest)
        
        if data:
            print(f"✅ {data['min_duration_minutes']} min")
            results.append(data)
        else:
            print(f"❌ No route")

    return pd.DataFrame(results)

In [26]:
routes_data = get_efficient_network_data(cities_stations['eva_id'].tolist())
routes_data

🚀 Optimized Search: 43 stations -> 1806 routes.
📅 Using representative weekday: Tuesday, 2026-01-06
[1/1806] 8103000 -> 8800004... 

KeyboardInterrupt: 

# Scrape API with intermittent saves to prevent data loss

In [35]:
# this code was generated by Claude (as you can tell by the comments lol)
# i went over it, fixed and added things, so it should be alright

import time

def scrape_train_routes_with_checkpoints(eva_ids: List[str], batch_size: int = 10):
    """
    Scrapes train routes and persists to CSV every batch_size API calls.
    Includes progress tracking and crash resilience.
    
    Args:
        eva_ids: List of station EVA IDs to query
        batch_size: Save to CSV after this many successful API calls (default: 10)
    """
    from itertools import permutations
    
    # Setup output file
    output_file = RAW_DATA_DIR / "train_routes_scraped.csv"
    
    # Check if file already exists to resume from where we left off
    processed_pairs = set()
    if output_file.exists():
        existing_df = pd.read_csv(output_file)
        processed_pairs = set(zip(existing_df['origin_id'], existing_df['destination_id']))
        print(f"📋 Resuming: Found {len(processed_pairs)} existing routes")
    else:
        print(f"📝 Starting fresh: Creating new output file at {output_file}")
    
    # Generate all pairs
    pairs = list(permutations(eva_ids, 2))
    pending_pairs = [p for p in pairs if p not in processed_pairs]
    
    print(f"🚀 Total routes to scrape: {len(pending_pairs)} (from {len(pairs)} possible pairs)")
    print(f"📅 Using representative weekday: {get_next_representative_weekday().strftime('%A, %Y-%m-%d')}\n")
    
    batch_results = []
    api_calls_count = 0
    failed_count = 0
    
    try:
        for idx, (origin, dest) in enumerate(pending_pairs):
            pair_num = idx + len(processed_pairs) + 1
            print(f"[{pair_num}/{len(pairs)}] {origin} → {dest}...", end=" ", flush=True)
            
            try:
                start_time = time.time()
                data = fetch_fastest_connection(origin, dest)
                elapsed = time.time() - start_time
                
                if data:
                    print(f"✅ {data['min_duration_minutes']} min (retrieved in {elapsed} seconds)")
                    batch_results.append(data)
                    api_calls_count += 1
                else:
                    print(f"⚠️  No route found (retrieved in {elapsed} seconds)")

                    # if no route was found, still add it to output dataset.
                    # this is so that the scraping does not try again for this city combination when restarted
                    data = {
                        "origin_id": origin,
                        "destination_id": dest,
                        "origin_name": "",
                        "destination_name": "",
                        "min_duration_minutes": -1,
                        "transfers": 0,
                        "trains": "",
                        "check_date": get_next_representative_weekday().strftime("%Y-%m-%d")
                    }
                    batch_results.append(data)

                    api_calls_count += 1
                    failed_count += 1
                
                # Batch save every N successful calls
                if api_calls_count >= batch_size:
                    batch_df = pd.DataFrame(batch_results)
                    
                    # Append to CSV (or create if doesn't exist)
                    if output_file.exists():
                        batch_df.to_csv(output_file, mode='a', header=False, index=False)
                    else:
                        batch_df.to_csv(output_file, mode='w', header=True, index=False)
                    
                    print(f"\n💾 Checkpoint: Saved {api_calls_count} routes to {output_file.name}")
                    
                    batch_results = []
                    api_calls_count = 0
                    
            except Exception as e:
                print(f"❌ Error: {e}")
                failed_count += 1
                time.sleep(2)  # Back off on error
                continue
        
        # Final save of remaining results
        if batch_results:
            batch_df = pd.DataFrame(batch_results)
            if output_file.exists():
                batch_df.to_csv(output_file, mode='a', header=False, index=False)
            else:
                batch_df.to_csv(output_file, mode='w', header=True, index=False)
            print(f"\n💾 Final Checkpoint: Saved {len(batch_results)} remaining routes")
        
        # Summary
        total_routes = len(processed_pairs) + len(batch_results) + api_calls_count
        print(f"\n" + "="*60)
        print(f"✨ Scraping Complete!")
        print(f"   Successful: {total_routes}")
        print(f"   Failed: {failed_count}")
        print(f"   Total pairs processed: {total_routes + failed_count}/{len(pairs)}")
        print(f"   Output file: {output_file}")
        print(f"="*60)
        
    except KeyboardInterrupt:
        print(f"\n⏸️  Interrupted by user")
        # Save any remaining batch data before exiting
        if batch_results:
            batch_df = pd.DataFrame(batch_results)
            if output_file.exists():
                batch_df.to_csv(output_file, mode='a', header=False, index=False)
            else:
                batch_df.to_csv(output_file, mode='w', header=True, index=False)
            print(f"💾 Saved {len(batch_results)} routes before exit")
        raise


In [36]:
# Start the scraping job
scrape_train_routes_with_checkpoints(cities_stations['eva_id'].tolist(), batch_size=10)

📋 Resuming: Found 273 existing routes
🚀 Total routes to scrape: 1806 (from 1806 possible pairs)
📅 Using representative weekday: Tuesday, 2026-01-06

[274/1806] 8103000 → 8800004... ✅ 622 min (retrieved in 3.0251519680023193 seconds)
[275/1806] 8103000 → 8800007... ✅ 653 min (retrieved in 3.5183348655700684 seconds)
[276/1806] 8103000 → 5400014... ✅ 253 min (retrieved in 1.4938020706176758 seconds)
[277/1806] 8103000 → 8011160... ✅ 429 min (retrieved in 1.148447036743164 seconds)
[278/1806] 8103000 → 8000096... ✅ 391 min (retrieved in 1.1549482345581055 seconds)
[279/1806] 8103000 → 8000261... ✅ 253 min (retrieved in 1.5753850936889648 seconds)
[280/1806] 8103000 → 8002549... ✅ 524 min (retrieved in 2.0647530555725098 seconds)
[281/1806] 8103000 → 8003368... ✅ 481 min (retrieved in 2.5753250122070312 seconds)
[282/1806] 8103000 → 8000105... ✅ 384 min (retrieved in 1.97574782371521 seconds)
[283/1806] 8103000 → 8000085... ✅ 499 min (retrieved in 2.96864914894104 seconds)

💾 Checkpoint: S

# Remove duplicates from saved data

In [5]:
import pandas as pd

input_file = f"{RAW_DATA_DIR}/train_routes_scraped.csv"
output_file = f"{PROCESSED_DATA_DIR}/train_routes_scraped_no_duplicates.csv"

df = pd.read_csv(input_file)

# Remove duplicate rows
df = df.drop_duplicates()

# Save result
df.to_csv(output_file, index=False)


# Add back cities to routes data

In [26]:
df_routes = pd.read_csv("Raw_Data/train_routes_scraped.csv")
df_routes.rename(columns={"origin_name": "origin_station_name", "destination_name": "destination_station_name"}, inplace=True)

cities_stations_names_only = cities_stations[["city_name", "station_name"]]

cities_stations_origin = cities_stations.add_prefix("origin_")
cities_stations_destination = cities_stations.add_prefix("destination_")

df_routes_with_city_names = df_routes.merge(cities_stations_origin, on="origin_station_name")
df_routes_with_city_names = df_routes_with_city_names.merge(cities_stations_destination, on="destination_station_name")

df_routes_with_city_names

,origin_id,destination_id,origin_station_name,destination_station_name,min_duration_minutes,transfers,trains,check_date,origin_city_name,origin_eva_id,origin_latitude,origin_longitude,destination_city_name,destination_eva_id,destination_latitude,destination_longitude
0,8103000,8000261,Wien Hbf,München Hbf,253,2,NaN,2026-01-06,Vienna,8103000,48.185101,16.377113,Munich,8000261,48.140232,11.558335
1,8103000,8002549,Wien Hbf,Hamburg Hbf,524,2,NaN,2026-01-06,Vienna,8103000,48.185101,16.377113,Hamburg,8002549,53.552736,10.006909
2,8103000,8000105,Wien Hbf,Frankfurt(Main)Hbf,384,0,NaN,2026-01-06,Vienna,8103000,48.185101,16.377113,Frankfurt,8000105,50.106682,8.662828
3,8103000,8000085,Wien Hbf,Düsseldorf Hbf,499,2,NaN,2026-01-06,Vienna,8103000,48.185101,16.377113,Düsseldorf,8000085,51.219962,6.794319
4,8103000,8010205,Wien Hbf,Leipzig Hbf,378,4,NaN,2026-01-06,Vienna,8103000,48.185101,16.377113,Leipzig,8010205,51.345471,12.382064
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2053,5600207,5100028,Bratislava hl.st.,Krakow Glowny,350,2,NaN,2026-01-06,Bratislava,5600207,48.158910,17.106466,Kraków,5100028,50.067194,19.947426
2054,5600207,5100069,Bratislava hl.st.,Wroclaw Glowny,380,4,NaN,2026-01-06,Bratislava,5600207,48.158910,17.106466,Wrocław,5100069,51.098078,17.037088
2055,5600207,5100081,Bratislava hl.st.,Poznan Glowny,407,2,NaN,2026-01-06,Bratislava,5600207,48.158910,17.106466,Poznań,5100081,52.401993,16.910870
2056,5600207,7400002,Bratislava hl.st.,Stockholm Central,1386,6,NaN,2026-01-06,Bratislava,5600207,48.158910,17.106466,Stockholm,7400002,59.330014,18.057630


In [28]:
df_routes_with_city_names.to_csv("Processed_Data/train_routes_scraped_processed.csv", index=False)